In [ ]:
!rm -rf ~/.cache/huggingface/evaluate

In [1]:
!pip install -q torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu128
!pip install -q transformers datasets seqeval evaluate tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00


In [2]:
# =========================================
# 2️⃣ Imports
# =========================================
import json
import re
import torch
import numpy as np
from tqdm import tqdm
import evaluate
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, pipeline
from sklearn.model_selection import train_test_split

seqeval_metric = evaluate.load("seqeval")

# =========================================
# 3️⃣ Load Dataset
# =========================================
dataset_path = "/content/cleaned_file.jsonl"

dataset = []
with open(dataset_path, "r") as f:
    for line in f:
        dataset.append(json.loads(line))

texts = [d["input"] for d in dataset]
labels = [json.loads(d["output"]) for d in dataset]

# =========================================
# 4️⃣ BIO Label Setup
# =========================================
entity_types = ["TOPIC", "CONCEPT", "METHOD", "ALGORITHM"]

label_list = ["O"]
for t in entity_types:
    label_list.append(f"B-{t}")
    label_list.append(f"I-{t}")

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {v: k for k, v in label2id.items()}

# =========================================
# 5️⃣ Text Normalization
# =========================================
def normalize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s\-]", "", text)
    return text

# =========================================
# 6️⃣ Encode Labels (BIO FIXED)
# =========================================
def encode_labels(text, entities):
    tokens = text.split()
    token_labels = ["O"] * len(tokens)

    for ent in entities:
        ent_text = normalize(ent["entity"])
        ent_tokens = ent_text.split()
        ent_label = ent["label"] if ent["label"] in entity_types else "CONCEPT"

        for i in range(len(tokens) - len(ent_tokens) + 1):
            if tokens[i:i+len(ent_tokens)] == ent_tokens:
                token_labels[i] = f"B-{ent_label}"
                for j in range(1, len(ent_tokens)):
                    token_labels[i+j] = f"I-{ent_label}"

    return token_labels

# Normalize texts
texts = [normalize(t) for t in texts]

token_labels = [encode_labels(t, e) for t, e in zip(texts, labels)]

# =========================================
# 7️⃣ Split Dataset
# =========================================
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, token_labels, test_size=0.15, random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.33, random_state=42
)

train_dataset = Dataset.from_dict({"tokens": [t.split() for t in train_texts], "ner_tags": train_labels})
val_dataset = Dataset.from_dict({"tokens": [t.split() for t in val_texts], "ner_tags": val_labels})
test_dataset = Dataset.from_dict({"tokens": [t.split() for t in test_texts], "ner_tags": test_labels})

# =========================================
# 8️⃣ Models
# =========================================
scibert_name = "allenai/scibert_scivocab_uncased"
roberta_name = "roberta-base"

scibert_tokenizer = AutoTokenizer.from_pretrained(scibert_name)
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_name)

scibert_model = AutoModelForTokenClassification.from_pretrained(scibert_name, num_labels=len(label_list))
roberta_model = AutoModelForTokenClassification.from_pretrained(roberta_name, num_labels=len(label_list))

for model in [scibert_model, roberta_model]:
    model.config.id2label = id2label
    model.config.label2id = label2id

# =========================================
# 9️⃣ Tokenization
# =========================================
def tokenize_and_align_labels(examples, tokenizer):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        max_length=128,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            else:
                label_ids.append(label2id[label[word_id]])

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

train_sci = train_dataset.map(lambda x: tokenize_and_align_labels(x, scibert_tokenizer), batched=True)
val_sci = val_dataset.map(lambda x: tokenize_and_align_labels(x, scibert_tokenizer), batched=True)

train_rob = train_dataset.map(lambda x: tokenize_and_align_labels(x, roberta_tokenizer), batched=True)
val_rob = val_dataset.map(lambda x: tokenize_and_align_labels(x, roberta_tokenizer), batched=True)

for d in [train_sci, val_sci, train_rob, val_rob]:
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# =========================================
# 🔟 Training
# =========================================
args = TrainingArguments(
    output_dir="./models",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    eval_strategy="epoch",   # ✅ FIXED
    save_strategy="epoch",
    logging_dir="./logs"
)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=2)
    labels = p.label_ids

    true = [[id2label[l] for l in lab if l != -100] for lab in labels]
    pred = [[id2label[p] for p, l in zip(pr, lab) if l != -100] for pr, lab in zip(preds, labels)]

    return seqeval_metric.compute(predictions=pred, references=true)

trainer_sci = Trainer(model=scibert_model, args=args, train_dataset=train_sci, eval_dataset=val_sci, compute_metrics=compute_metrics)
trainer_rob = Trainer(model=roberta_model, args=args, train_dataset=train_rob, eval_dataset=val_rob, compute_metrics=compute_metrics)

trainer_sci.train()
trainer_rob.train()

# =========================================
# 1️⃣1️⃣ Pipelines (FIXED)
# =========================================
sci_pipe = pipeline(
    "ner",
    model=scibert_model,
    tokenizer=scibert_tokenizer,
    aggregation_strategy="simple"
)

rob_pipe = pipeline(
    "ner",
    model=roberta_model,
    tokenizer=roberta_tokenizer,
    aggregation_strategy="simple"
)
# =========================================
# 1️⃣2️⃣ Strong Filtering
# =========================================
STOPWORDS = {"the", "and", "or", "to", "of", "in", "a"}

def clean_entity(text):
    text = text.lower().strip()
    if text in STOPWORDS:
        return None
    if len(text) < 3:
        return None
    if text.endswith(("and", "or")):
        return None
    if "," in text:
        return None
    return text

# =========================================
# 1️⃣3️⃣ Ensemble (IMPROVED)
# =========================================
def ensemble(sci, rob):
    combined = {}

    for e in sci:
        key = clean_entity(e["word"])
        if not key: continue
        combined[key] = {"label": e["entity_group"], "score": e["score"], "source": "sci"}

    for e in rob:
        key = clean_entity(e["word"])
        if not key: continue

        if key in combined:
            combined[key]["score"] = max(combined[key]["score"], e["score"])
            combined[key]["source"] = "both"
        else:
            if e["score"] > 0.75:
                combined[key] = {"label": e["entity_group"], "score": e["score"], "source": "rob"}

    # keep strong ones
    final = []
    for k, v in combined.items():
        if v["source"] == "both" or v["score"] > 0.8:
            final.append({
                "entity": k,
                "label": v["label"],
                "score": round(float(v["score"]), 3)
            })

    return final

# =========================================
# 🔹 Chunking الطويل texts (VERY IMPORTANT)
# =========================================
def chunk_text(text, max_words=100):
    words = text.split()
    chunks = []

    for i in range(0, len(words), max_words):
        chunk = " ".join(words[i:i+max_words])
        chunks.append(chunk)

    return chunks

# =========================================
# 1️⃣4️⃣ Run Ensemble
# =========================================
results = []

for text in tqdm(test_texts):
    sci_all, rob_all = [], []

    for chunk in chunk_text(text):
        sci_all.extend(sci_pipe(chunk))
        rob_all.extend(rob_pipe(chunk))

    ents = ensemble(sci_all, rob_all)

    results.append({
        "text": text,
        "entities": ents
    })
# =========================================
# 1️⃣5️⃣ Save
# =========================================
with open("FINAL_KG_READY.json", "w") as f:
    json.dump(results, f, indent=4)

print("✅ CLEAN KG FILE READY")

# =========================================
# 📊 EVALUATION CELL (TEST SET)
# =========================================

def evaluate_model(trainer, dataset, name="Model"):
    preds_output = trainer.predict(dataset)

    preds = np.argmax(preds_output.predictions, axis=2)
    labels = preds_output.label_ids

    true_labels = [
        [id2label[l] for l in lab if l != -100]
        for lab in labels
    ]
    pred_labels = [
        [id2label[p] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(preds, labels)
    ]

    results = seqeval_metric.compute(
        predictions=pred_labels,
        references=true_labels
    )

    print(f"\n🔹 {name} Evaluation Results:")
    print(f"Precision : {results['overall_precision']:.4f}")
    print(f"Recall    : {results['overall_recall']:.4f}")
    print(f"F1 Score  : {results['overall_f1']:.4f}")
    print(f"Accuracy  : {results['overall_accuracy']:.4f}")


# Tokenize test set
test_sci = test_dataset.map(lambda x: tokenize_and_align_labels(x, scibert_tokenizer), batched=True)
test_rob = test_dataset.map(lambda x: tokenize_and_align_labels(x, roberta_tokenizer), batched=True)

for d in [test_sci, test_rob]:
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# Run evaluation
evaluate_model(trainer_sci, test_sci, "SciBERT")
evaluate_model(trainer_rob, test_rob, "RoBERTa")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

✅ CLEAN KG FILE READY


Map:   0%|          | 0/227 [00:00<?, ? examples/s]

Map:   0%|          | 0/227 [00:00<?, ? examples/s]


🔹 SciBERT Evaluation Results:
Precision : 0.9601
Recall    : 0.9803
F1 Score  : 0.9701
Accuracy  : 0.9942



🔹 RoBERTa Evaluation Results:
Precision : 0.9338
Recall    : 0.9736
F1 Score  : 0.9532
Accuracy  : 0.9885


In [7]:
# =========================================
# 📊 ENSEMBLE EVALUATION (FULL FIXED CLEAN VERSION)
# =========================================
import numpy as np
from tqdm import tqdm
import evaluate

seqeval_metric = evaluate.load("seqeval")

# =========================================
# 🔧 BIO → ENTITY CONVERSION
# =========================================
def bio_to_entities(tokens, labels):
    entities = []
    i = 0

    while i < len(labels):
        if labels[i].startswith("B-"):
            label_type = labels[i][2:]
            entity_tokens = [tokens[i]]

            j = i + 1
            while j < len(labels) and labels[j] == f"I-{label_type}":
                entity_tokens.append(tokens[j])
                j += 1

            entities.append({
                "entity": " ".join(entity_tokens),
                "label": label_type
            })

            i = j
        else:
            i += 1

    return entities


# =========================================
# 📊 ENSEMBLE EVALUATION FUNCTION
# =========================================
def evaluate_ensemble(texts, true_bio_labels):
    all_preds = []
    all_truths = []
    all_ents = []

    for text, true_labels in tqdm(zip(texts, true_bio_labels), total=len(texts)):

        tokens = text.split()

        # =========================================
        # 1️⃣ Ground truth BIO → entities
        # =========================================
        true_entities = bio_to_entities(tokens, true_labels)

        # Convert back to BIO (GROUND TRUTH)
        true_bio = encode_labels(text, true_entities)
        true_bio_str = true_bio   # already BIO strings (NO id2label!)

        # =========================================
        # 2️⃣ Ensemble prediction
        # =========================================
        sci_all, rob_all = [], []

        for chunk in chunk_text(text):
            sci_all.extend(sci_pipe(chunk))
            rob_all.extend(rob_pipe(chunk))

        ents = ensemble(sci_all, rob_all)
        all_ents.append(ents)

        # =========================================
        # 3️⃣ Pred → BIO
        # =========================================
        pred_bio = encode_labels(text, ents)
        pred_bio_str = pred_bio   # already BIO strings (NO id2label!)

        # =========================================
        # 4️⃣ Align lengths
        # =========================================
        min_len = min(len(pred_bio_str), len(true_bio_str))

        all_preds.append(pred_bio_str[:min_len])
        all_truths.append(true_bio_str[:min_len])

    # =========================================
    # 5️⃣ METRICS
    # =========================================
    results = seqeval_metric.compute(
        predictions=all_preds,
        references=all_truths
    )

    print("\n📊 ENSEMBLE Evaluation Results:")
    print(f"Precision : {results['overall_precision']:.4f}")
    print(f"Recall    : {results['overall_recall']:.4f}")
    print(f"F1 Score  : {results['overall_f1']:.4f}")
    print(f"Accuracy  : {results['overall_accuracy']:.4f}")

    return results, all_ents


# =========================================
# 🚀 RUN EVALUATION
# =========================================

# Normalize test texts (same as training)
test_texts_norm = [normalize(t) for t in test_texts]

results, ensemble_predictions = evaluate_ensemble(
    test_texts_norm,
    test_labels
)

100%|██████████| 227/227 [00:08<00:00, 26.11it/s]



📊 ENSEMBLE Evaluation Results:
Precision : 0.8559
Recall    : 0.8301
F1 Score  : 0.8428
Accuracy  : 0.9745


Evaluation on SE

In [11]:
# =========================================
# 🧪 TEST + ENSEMBLE EVALUATION (CLEAN FIXED VERSION)
# =========================================
import json
import numpy as np
from tqdm import tqdm
import evaluate

seqeval_metric = evaluate.load("seqeval")

test_file_path = "/content/SE-cleaned_file.jsonl"

# =========================================
# 1️⃣ Load Data
# =========================================
dataset = []
with open(test_file_path, "r") as f:
    for line in f:
        dataset.append(json.loads(line))

texts = [normalize(d["input"]) for d in dataset]
true_entities = [json.loads(d["output"]) for d in dataset]

# =========================================
# 2️⃣ Ground Truth → BIO labels
# =========================================
true_token_labels = [encode_labels(t, e) for t, e in zip(texts, true_entities)]

# IMPORTANT FIX:
# already BIO strings → DO NOT use id2label
true_labels_str = true_token_labels

# =========================================
# 3️⃣ ENSEMBLE PREDICTIONS
# =========================================
all_pred_labels = []
all_ents = []

for text in tqdm(texts):
    sci_all, rob_all = [], []

    for chunk in chunk_text(text):
        sci_all.extend(sci_pipe(chunk))
        rob_all.extend(rob_pipe(chunk))

    ents = ensemble(sci_all, rob_all)
    all_ents.append(ents)

    # BIO prediction (already strings)
    pred_labels = encode_labels(text, ents)
    pred_labels_str = pred_labels   # NO id2label

    all_pred_labels.append(pred_labels_str)

# =========================================
# 4️⃣ Align lengths
# =========================================
aligned_preds = []
aligned_truths = []

for pred, true in zip(all_pred_labels, true_labels_str):
    min_len = min(len(pred), len(true))
    aligned_preds.append(pred[:min_len])
    aligned_truths.append(true[:min_len])

# =========================================
# 5️⃣ Evaluation
# =========================================
results = seqeval_metric.compute(
    predictions=aligned_preds,
    references=aligned_truths
)

print("\n📊 ENSEMBLE RESULTS ON NEW FILE:")
print(f"Precision : {results['overall_precision']:.4f}")
print(f"Recall    : {results['overall_recall']:.4f}")
print(f"F1 Score  : {results['overall_f1']:.4f}")
print(f"Accuracy  : {results['overall_accuracy']:.4f}")

# =========================================
# 6️⃣ Save Predictions
# =========================================
output = []

for text, ents in zip(texts, all_ents):
    output.append({
        "text": text,
        "predicted_entities": ents
    })

with open("NEW_FILE_PREDICTIONS.json", "w") as f:
    json.dump(output, f, indent=4)

print("\n✅ Saved: NEW_FILE_PREDICTIONS.json")

100%|██████████| 30/30 [00:00<00:00, 40.40it/s]


📊 ENSEMBLE RESULTS ON NEW FILE:
Precision : 0.6190
Recall    : 0.0844
F1 Score  : 0.1486
Accuracy  : 0.8210

✅ Saved: NEW_FILE_PREDICTIONS.json



/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
!rm -rf /content/roberta-relation

traing data relation


In [12]:
# =========================================
# 1️⃣ Install Libraries
# =========================================
!pip install -q transformers datasets scikit-learn

# =========================================
# 2️⃣ Imports
# =========================================
import json
import torch
import numpy as np
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
    logging
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

logging.set_verbosity_error()

# =========================================
# 3️⃣ Load RE Dataset
# =========================================
with open("/content/All-relations.json", encoding="latin-1") as f:
    data = json.load(f)

texts = [d["input"] for d in data]
labels = [d["output"].strip() for d in data]

label_list = ["used_for", "based_on", "implements", "part_of", "improves", "no_relation"]
label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for l,i in label2id.items()}

labels = [l if l in label2id else "no_relation" for l in labels]

# =========================================
# 4️⃣ Train / Val / Test Split
# =========================================
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, labels, test_size=0.1, random_state=42, stratify=labels
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
)

def make_dataset(texts, labels):
    return Dataset.from_dict({
        "text": texts,
        "label": [label2id[l] for l in labels]
    })

train_dataset = make_dataset(train_texts, train_labels)
val_dataset = make_dataset(val_texts, val_labels)
test_dataset = make_dataset(test_texts, test_labels)

# =========================================
# 5️⃣ Tokenization
# =========================================
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=256)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

for d in [train_dataset, val_dataset, test_dataset]:
    d.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# =========================================
# 6️⃣ Model
# =========================================
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# =========================================
# 7️⃣ Metrics
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# =========================================
# 8️⃣ Training Args
# =========================================
training_args = TrainingArguments(
    output_dir="./roberta-relation",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    learning_rate=2e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

# =========================================
# 9️⃣ Trainer
# =========================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# =========================================
# 🔟 Train
# =========================================
trainer.train()

# =========================================
# 🧪 TEST SET EVALUATION
# =========================================
test_metrics = trainer.evaluate(eval_dataset=test_dataset)

print("\n📊 TEST SET RESULTS:")
print(f"Accuracy : {test_metrics['eval_accuracy']:.4f}")
print(f"Precision: {test_metrics['eval_precision']:.4f}")
print(f"Recall   : {test_metrics['eval_recall']:.4f}")
print(f"F1 Score : {test_metrics['eval_f1']:.4f}")

# =========================================
# 1️⃣1️⃣ Load NER Output
# =========================================
with open("/content/FINAL_KG_READY.json") as f:
    test_data = json.load(f)

# =========================================
# 1️⃣2️⃣ Clean Entities
# =========================================
def clean_entities(entities):
    cleaned = []
    for e in entities:
        text = e["entity"].strip().lower()

        if len(text) < 4:
            continue
        if text in ["the", "this", "that", "data"]:
            continue
        if e["label"] == "OTHER":
            continue

        cleaned.append(e)

    return cleaned[:5]   # 🔥 LIMIT TO TOP 5

# =========================================
# 1️⃣3️⃣ Direction-Aware Prediction
# =========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def predict_best_relation(e1, t1, e2, t2, text):
    def run(inp):
        inputs = tokenizer(inp, return_tensors="pt", truncation=True, padding=True).to(device)
        with torch.no_grad():
            out = model(**inputs)
            probs = torch.softmax(out.logits, dim=1)
            conf, pred = torch.max(probs, dim=1)
        return id2label[pred.item()], conf.item()

    input1 = f"Sentence: {text} Entity1: {e1} ({t1}). Entity2: {e2} ({t2})."
    input2 = f"Sentence: {text} Entity1: {e2} ({t2}). Entity2: {e1} ({t1})."

    r1, c1 = run(input1)
    r2, c2 = run(input2)

    if c1 >= c2:
        return e1, t1, r1, e2, t2, c1
    else:
        return e2, t2, r2, e1, t1, c2

# =========================================
# 1️⃣4️⃣ Predict Relations
# =========================================
triplets = []
seen = set()

for item in test_data:
    text = item["text"]
    entities = clean_entities(item.get("entities", []))

    for i in range(len(entities)):
        for j in range(i+1, len(entities)):

            e1 = entities[i]["entity"]
            t1 = entities[i]["label"]
            e2 = entities[j]["entity"]
            t2 = entities[j]["label"]

            if e1.lower() == e2.lower():
                continue

            e1, t1, rel, e2, t2, conf = predict_best_relation(e1, t1, e2, t2, text)

            # 🔥 Strong filtering
            if rel == "no_relation" or conf < 0.8:
                continue

            key = (e1.lower(), rel, e2.lower())
            if key in seen:
                continue
            seen.add(key)

            triplets.append({
                "text": text,
                "entity1": e1,
                "entity1_type": t1,
                "relation": rel,
                "entity2": e2,
                "entity2_type": t2,
                "confidence": round(conf, 3)
            })

# =========================================
# 1️⃣5️⃣ Save Output
# =========================================
with open("final_triplets_level2.json", "w") as f:
    json.dump(triplets, f, indent=4)

print("\n✅ FINAL LEVEL-2 triplets saved to final_triplets_level2.json")

Map:   0%|          | 0/5283 [00:00<?, ? examples/s]

Map:   0%|          | 0/294 [00:00<?, ? examples/s]

Map:   0%|          | 0/294 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

{'loss': '1.117', 'grad_norm': '15.58', 'learning_rate': '1.698e-05', 'epoch': '0.7564'}
{'eval_loss': '0.721', 'eval_accuracy': '0.7789', 'eval_precision': '0.7971', 'eval_recall': '0.7789', 'eval_f1': '0.763', 'eval_runtime': '3.941', 'eval_samples_per_second': '74.6', 'eval_steps_per_second': '9.389', 'epoch': '1'}


...

{'loss': '0.5085', 'grad_norm': '33.14', 'learning_rate': '7.903e-06', 'epoch': '3.026'}
{'loss': '0.3892', 'grad_norm': '56.25', 'learning_rate': '4.877e-06', 'epoch': '3.782'}
{'eval_loss': '0.5747', 'eval_accuracy': '0.8503', 'eval_precision': '0.8562', 'eval_recall': '0.8503', 'eval_f1': '0.852', 'eval_runtime': '4.028', 'eval_samples_per_second': '72.99', 'eval_steps_per_second': '9.185', 'epoch': '4'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.3539', 'grad_norm': '54.15', 'learning_rate': '1.852e-06', 'epoch': '4.539'}
{'eval_loss': '0.6106', 'eval_accuracy': '0.8605', 'eval_precision': '0.8629', 'eval_recall': '0.8605', 'eval_f1': '0.8613', 'eval_runtime': '4.03', 'eval_samples_per_second': '72.96', 'eval_steps_per_second': '9.182', 'epoch': '5'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '1286', 'train_samples_per_second': '20.54', 'train_steps_per_second': '2.57', 'train_loss': '0.6094', 'epoch': '5'}
{'eval_loss': '0.713', 'eval_accuracy': '0.8435', 'eval_precision': '0.8419', 'eval_recall': '0.8435', 'eval_f1': '0.8418', 'eval_runtime': '3.865', 'eval_samples_per_second': '76.06', 'eval_steps_per_second': '9.572', 'epoch': '5'}

📊 TEST SET RESULTS:
Accuracy : 0.8435
Precision: 0.8419
Recall   : 0.8435
F1 Score : 0.8418

✅ FINAL LEVEL-2 triplets saved to final_triplets_level2.json


SE relatio evaluation

In [21]:
# =========================================
# 🧪 RELATION EVAL + PREDICTION PIPELINE
# =========================================

import json
import numpy as np
import torch
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# =========================================
# 1️⃣ LOAD RELATION MODEL (already trained)
# =========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# =========================================
# 2️⃣ LABEL MAP
# =========================================
# (already defined in your training code)
# label2id, id2label, tokenizer must exist

# =========================================
# 3️⃣ LOAD DATASETS
# =========================================

# evaluation dataset (GROUND TRUTH)
with open("/content/SE_textDocx_relations.json") as f:
    eval_data = json.load(f)

# entity prediction file (NER output)
with open("/content/NEW_FILE_PREDICTIONS.json") as f:
    ner_data = json.load(f)

# =========================================
# 4️⃣ CLEAN ENTITIES
# =========================================
def clean_entities(entities):
    cleaned = []
    for e in entities:
        text = e["entity"].strip().lower()
        if len(text) < 3:
            continue
        if text in ["the", "this", "that", "data"]:
            continue
        cleaned.append(e)
    return cleaned

# =========================================
# 5️⃣ RELATION PREDICT FUNCTION
# =========================================
def predict_relation(text, e1, t1, e2, t2):
    inp = f"Sentence: {text} Entity1: {e1} ({t1}). Entity2: {e2} ({t2})."

    inputs = tokenizer(
        inp,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        out = model(**inputs)
        probs = torch.softmax(out.logits, dim=1)
        conf, pred = torch.max(probs, dim=1)

    return id2label[pred.item()]

# =========================================
# 6️⃣ EVALUATION LOOP
# =========================================
y_true = []
y_pred = []

for eval_item, ner_item in tqdm(zip(eval_data, ner_data), total=len(eval_data)):

    text = eval_item["input"]
    gold_rel = eval_item["output"].strip()

    entities = clean_entities(ner_item["predicted_entities"])

    predicted_rel = "no_relation"

    # build entity pairs
    for i in range(len(entities)):
        for j in range(i + 1, len(entities)):

            e1 = entities[i]["entity"]
            t1 = entities[i]["label"]
            e2 = entities[j]["entity"]
            t2 = entities[j]["label"]

            rel = predict_relation(text, e1, t1, e2, t2)

            if rel != "no_relation":
                predicted_rel = rel
                break

        if predicted_rel != "no_relation":
            break

    y_true.append(gold_rel)
    y_pred.append(predicted_rel)

# =========================================
# 7️⃣ METRICS
# =========================================
labels = sorted(list(set(y_true + y_pred)))

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=labels, average="weighted", zero_division=0
)

acc = accuracy_score(y_true, y_pred)

print("\n📊 RELATION EVALUATION RESULTS")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

# =========================================
# 8️⃣ SAVE FINAL RELATION PREDICTIONS
# =========================================
output = []

for eval_item, ner_item, pred in zip(eval_data, ner_data, y_pred):

    output.append({
        "text": eval_item["input"],
        "gold_relation": eval_item["output"],
        "predicted_relation": pred
    })

with open("FINAL_RELATION_RESULTS.json", "w") as f:
    json.dump(output, f, indent=4)

print("\n✅ Saved: FINAL_RELATION_RESULTS.json")

100%|██████████| 26/26 [00:00<00:00, 567.39it/s]


📊 RELATION EVALUATION RESULTS
Accuracy : 0.2692
Precision: 0.2092
Recall   : 0.2692
F1 Score : 0.1509

✅ Saved: FINAL_RELATION_RESULTS.json
